# RepairScopeX NB29E — New Candidate Discovery + Parser Audit (144-record continuation)

This is **not a repeat notebook**. It includes your 144-record snapshot internally and restores only records absent from Drive. It audits prior ambiguous outputs *without changing any recorded ground-truth labels*, then advances to untried candidates using the original fixed task bank, model, protocol and grader.

The previous ~54% unanswered/ambiguous rate may be partly caused by output truncation; this notebook explicitly audits that question before allocating new GPU generations. A CPU-only audit would be insufficient to establish new failures; this notebook continues the existing fixed 648-problem bank and stops when exhausted.

**Run all** in Colab with T4 GPU. Checkpoints remain under your existing V1 Drive folder. No existing trial IDs are regenerated. The frozen cohort gate is unchanged.

**Scope:** V1 protocol only (not V3 Pilot256). This notebook contains user-supplied records, not manually relabeled results.


In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX29'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned official ALoDLM source:',CODE_COMMIT)

In [ ]:
import os, sys, gc, json, csv, re, math, ast, time, random, hashlib, zipfile, tempfile, statistics, shutil, platform, traceback, subprocess
from datetime import datetime,timezone
from pathlib import Path
from collections import defaultdict,Counter
from dataclasses import dataclass
import torch, numpy as np, pandas as pd, transformers
from scipy.stats import binomtest, beta
from huggingface_hub import snapshot_download
from types import SimpleNamespace
from google.colab import drive, files
import matplotlib.pyplot as plt
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('REQUIRES a real GPU: Colab > Runtime > Change runtime type > T4 GPU.')
if torch.cuda.get_device_properties(0).total_memory < 12*2**30: raise RuntimeError('Needs at least 12 GiB GPU VRAM.')
if torch.cuda.memory_allocated() > 2*2**30: raise RuntimeError('Restart runtime: unexpected other GPU allocations.')
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/RepairScopeX/NB29_SemanticError_FP32_V1')
ROOT.mkdir(parents=True,exist_ok=True)
ART=ROOT/'artifacts'; ART.mkdir(exist_ok=True)
CACHE=ROOT/'records'; CACHE.mkdir(exist_ok=True)
MAX_SESSION_MINUTES=160
SESSION_START=time.monotonic()
MODEL_ID='amazon/ALoDLM-1.7B'
MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
PRECISION='fp32'; Q=0.4; TAU=0.4; WINDOW=16; MAX_NEW=96
GEN_MODE='left1'; REPAIR_MODE='entropy'
BASE_SEED=992941
CAL_TARGET={'countdown':4,'arithmetic':4,'multistep':4}
TEST_TARGET={'countdown':8,'arithmetic':8,'multistep':8}
ACTIONS=('global','tail25','tail50','tail75')
CAL_SEEDS=(23001,23002)
TEST_SEEDS=(24001,24002,24003)
STAGED_SEEDS=(25001,25002)
ABLATION_SEEDS=(26001,26002)
POLICIES=('global','best_fixed','adaptive','random_scope','tail50')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('Torch',torch.__version__,'Transformers',transformers.__version__)

def canonical(v): return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'),default=str)
def sha(v): return hashlib.sha256((v if isinstance(v,bytes) else canonical(v).encode())).hexdigest()
def atomic_write(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w',encoding='utf8') as h:
        h.write(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2,default=str)+'\n');h.flush();os.fsync(h.fileno())
    os.replace(tmp,path)
def rec_path(stage,tid):return CACHE/stage/(sha(tid)+'.json')
def save_rec(stage,rec):
    path=rec_path(stage,rec['trial_id'])
    if path.exists():
        old=json.loads(path.read_text());
        if sha(old)!=sha(rec):raise RuntimeError('Refusing to overwrite a different completed record: '+rec['trial_id'])
        return
    atomic_write(path,rec)
def get_rec(stage,tid):
    p=rec_path(stage,tid)
    if not p.exists():return None
    r=json.loads(p.read_text())
    if r['trial_id']!=tid or r.get('protocol_hash')!=RUN_SIG:raise RuntimeError('Checkpoint signature or identity mismatch '+tid)
    return r
def all_recs(stage):
    out=[]
    for p in sorted((CACHE/stage).glob('*.json')):
        r=json.loads(p.read_text())
        assert r['protocol_hash']==RUN_SIG
        out.append(r)
    assert len({r['trial_id'] for r in out})==len(out)
    return out
def stop_if_near_budget():
    used=(time.monotonic()-SESSION_START)/60
    if used >= MAX_SESSION_MINUTES:raise RuntimeError(f'Session wall-time safety stop ({used:.1f}/{MAX_SESSION_MINUTES} min). Export partial ZIP and resume in a new Colab session; earlier records are saved.')

PROTOCOL={'version':'RSX29-AloDLM-semantic-wrong-v1','model':MODEL_ID,'model_rev':MODEL_REV,'code_commit':CODE_COMMIT,
          'fp':PRECISION,'q':Q,'tau':TAU,'window':WINDOW,'max_new':MAX_NEW,
          'gen_mode':GEN_MODE,'repair_mode':REPAIR_MODE,'cal_target':CAL_TARGET,'test_target':TEST_TARGET,
          'actions':ACTIONS,'cal_seeds':CAL_SEEDS,'test_seeds':TEST_SEEDS,
          'staged_seeds':STAGED_SEEDS,'ablation_seeds':ABLATION_SEEDS,'base_seed':BASE_SEED}
RUN_SIG=sha(PROTOCOL)
manifest=ROOT/'protocol.json'
if manifest.exists():
    existing=json.loads(manifest.read_text())
    if existing['hash']!=RUN_SIG or sha(existing['protocol'])!=sha(PROTOCOL):raise RuntimeError('Different experiment in this Drive path. NEVER overwrite or mix protocols. Use a new folder.')
else:atomic_write(manifest,{'hash':RUN_SIG,'protocol':PROTOCOL,'created_utc':datetime.now(timezone.utc).isoformat()})

In [ ]:
# Embedded 144-record checkpoint: restore MERGE ONLY; never overwrite completed work.
import io,base64
_EMBEDDED_SNAPSHOT_B64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_EMBEDDED_SNAPSHOT_B64))) as archive:
    manifest=json.loads(archive.read('protocol.json'))
    if manifest['hash']!=RUN_SIG or sha(manifest['protocol'])!=RUN_SIG:
        raise RuntimeError('Protocol mismatch; no checkpoint data imported.')
    names=[x for x in archive.namelist() if x.startswith('records/natural/') and x.endswith('.json')]
    assert len(names)==144, f'Expected 144 saved records, found {len(names)}'
    imported=0; already=0
    for name in names:
        rec=json.loads(archive.read(name))
        assert rec['protocol_hash']==RUN_SIG
        target=rec_path('natural',rec['trial_id'])
        assert target.name==name.rsplit('/',1)[-1]
        if target.exists():
            if json.loads(target.read_text())!=rec:
                raise RuntimeError('Saved record conflicts with bundled snapshot: '+rec['trial_id'])
            already+=1
        else:
            save_rec('natural',rec); imported+=1
    print('Checkpoint merged:',imported,'records imported,',already,'already on Drive; total',len(all_recs('natural')))


In [ ]:
def make_problem(kind, rng, i, split):
    # Derived from deterministic synthetic templates, not GSM8K or SVAMP. No LLM-generated labels.
    if kind=='countdown':
        a,b,c=[rng.randint(3,30) for _ in range(3)]
        op=rng.choice(('plusminus','sum','prodminus','twice'))
        if op=='plusminus':target=a+b-c
        elif op=='sum':target=a+b+c
        elif op=='prodminus':target=a*b-c
        else:target=a*(b-c)
        prompt=f'Use the numbers {a}, {b}, and {c} exactly once to produce the target {target}. Return a valid arithmetic expression and end with `FINAL_ANSWER: <number>`.'
        return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[a,b,c],'template':op}
    if kind=='arithmetic':
        mode=i%4
        a=rng.randint(18,100);b=rng.randint(4,30)
        if mode==0:
            target=a+b;prompt=f'A collection contains {a} coins. A friend adds {b} more coins. How many coins are there now? End with FINAL_ANSWER: <number>.'
        elif mode==1:
            target=a-b;prompt=f'A shelf holds {a} books. Exactly {b} books are borrowed. How many books remain? End with FINAL_ANSWER: <number>.'
        elif mode==2:
            target=a*b;prompt=f'There are {a} identical boxes, each containing {b} pens. How many pens are there altogether? End with FINAL_ANSWER: <number>.'
        else:
            target=a;prompt=f'A number of stickers is divided equally among {b} children, and every child receives {a} stickers. How many stickers were there originally? End with FINAL_ANSWER: <number>.';target=a*b
        return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[],'template':mode}
    mode=i%4
    if mode==0:
        a=rng.randint(20,90);b=rng.randint(3,10);c=rng.randint(5,24);d=rng.randint(1,18)
        target=a+b*c-d;prompt=f'A warehouse begins with {a} packages. It receives {b} cartons with {c} packages each, then ships {d} packages. How many packages remain? End with FINAL_ANSWER: <number>.'
    elif mode==1:
        a=rng.randint(5,18);b=rng.randint(4,17);c=rng.randint(2,10);d=rng.randint(1,10)
        target=a*b+c*d;prompt=f'A school buys {a} packets containing {b} pencils each and {c} packets containing {d} pencils each. How many pencils in total? End with FINAL_ANSWER: <number>.'
    elif mode==2:
        a=rng.randint(3,12);b=rng.randint(5,20);c=rng.randint(3,25)
        target=a*b-c;prompt=f'A gardener plants {a} rows with {b} flowers each. Later {c} flowers are removed. How many flowers remain? End with FINAL_ANSWER: <number>.'
    else:
        a=rng.randint(10,40);b=rng.randint(2,8);c=rng.randint(5,20)
        target=a*b+c;prompt=f'A shop has {a} cartons of {b} bottles. A delivery adds {c} single bottles. How many bottles are available? End with FINAL_ANSWER: <number>.'
    return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[],'template':mode}

GLOBAL_FINGERPRINTS=set()
def build_bank(split,n_per_kind):
    out=[];seen=set()
    for k_idx,kind in enumerate(('countdown','arithmetic','multistep')):
        for i in range(n_per_kind):
            for retry in range(200):
                seed=BASE_SEED+(0 if split=='calibration' else 1000000)+k_idx*10000+i*11+retry*310009
                rng=random.Random(seed)
                d=make_problem(kind,rng,i,split)
                fp=sha({'prompt':d['prompt'],'kind':kind})
                if fp not in seen and fp not in GLOBAL_FINGERPRINTS:
                    seen.add(fp);GLOBAL_FINGERPRINTS.add(fp)
                    d.update(split=split,problem_id=f'rsx29:{split}:{kind}:{i:04d}',
                             problem_fingerprint=fp,collision_retry=retry)
                    out.append(d)
                    break
            else:raise RuntimeError('Could not generate distinct frozen task within 200 deterministic retries.')
    return out
CAL_BANK=build_bank('calibration',88)
TEST_BANK=build_bank('heldout',128)
assert len(set(r['problem_fingerprint'] for r in CAL_BANK+TEST_BANK))==len(CAL_BANK+TEST_BANK)
BANK_HASH=sha(CAL_BANK+TEST_BANK)
BANK_PATH=ROOT/'frozen_problem_bank.json'
if BANK_PATH.exists():assert json.loads(BANK_PATH.read_text())['hash']==BANK_HASH
else:atomic_write(BANK_PATH,{'hash':BANK_HASH,'problems':CAL_BANK+TEST_BANK})
print('Frozen',len(CAL_BANK),'calibration candidates and',len(TEST_BANK),'heldout candidates; checksum',BANK_HASH)

In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')


In [ ]:
# No retroactive reclassification: independent audit of why older outputs were ambiguous.
records=all_recs('natural')
reason=Counter(); stop=Counter(); split_kind=Counter(); questionable=[]
for r in records:
    e=r.get('evaluation',{})
    reason[e.get('failure_type','missing')]+=1
    stop[(e.get('failure_type','missing'),r.get('stop_reason','missing'))]+=1
    split_kind[(r.get('split'),r.get('kind'),e.get('failure_type'))]+=1
    if e.get('failure_type')=='unanswered_or_ambiguous':
        text=r.get('original_output','')
        has_numeric=bool(re.search(r'(?im)^\s*FINAL_ANSWER\s*:',text))
        questionable.append({'trial_id':r['trial_id'],'split':r['split'],'kind':r['kind'],
            'stop':r.get('stop_reason'), 'chars':len(text),'has_final_answer_marker':has_numeric,
            'tail':text[-160:].replace('\n',' ')})
print('Saved records:',len(records));print('Classifications:',dict(reason))
print('Stop by classification:',dict(stop))
print('Breakdown:')
for k,v in sorted(split_kind.items()):print(k,v)
report={'protocol_hash':RUN_SIG,'record_count':len(records),'classifications':dict(reason),
        'stop_by_classification':{'|'.join(map(str,k)):v for k,v in stop.items()},
        'split_kind':{'|'.join(map(str,k)):v for k,v in split_kind.items()},
        'ambiguous_samples':questionable}
atomic_write(ART/'NB29E_parser_diagnostics.json',report)
print('Example ambiguous tails (inspect; not automatically regraded):')
for x in questionable[:8]:print(x['kind'],x['stop'],x['tail'])


In [ ]:
sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
assert MODEL_REV in str(MODEL_DIR) or Path(MODEL_DIR).is_dir()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
decoder=load_decoder(args)
model=decoder.model;tokenizer=decoder.tokenizer
assert next(model.parameters()).dtype==torch.float32
print('Model loaded FP32; CUDA allocated GiB:',round(torch.cuda.memory_allocated()/2**30,2))
PROMPT_SUFFIX='\n\nSolve carefully. For Countdown, show an arithmetic expression using all given numbers exactly once. End your response with FINAL_ANSWER: <number>.'
def encode_prompt(p):
    return list(tokenizer.apply_chat_template([{'role':'user','content':p['prompt']+PROMPT_SUFFIX}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def decode_ids(ids):return tokenizer.decode(ids,skip_special_tokens=True).strip()
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
@torch.inference_mode()
def invoke(prompt_ids,new_tokens,mode,seed):
    assert new_tokens>0 and new_tokens<=MAX_NEW
    stop_if_near_budget();set_seed(int(seed))
    cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=int(new_tokens),position_penalty=None)
    cfg.validate()
    torch.cuda.reset_peak_memory_stats()
    started=time.perf_counter();result=decoder.generate(list(prompt_ids),cfg)
    ids=list(map(int,result['token_ids']))
    return {'token_ids':ids,'generated_tokens':int(result.get('generated_tokens',len(ids))),
            'outer_steps':int(result['outer_steps']),'recurrent_passes':int(result['recurrent_passes']),
            'loop_token_passes':int(result['loop_token_passes']),
            'mean_exit_depth':float(result['mean_exit_depth']) if result.get('mean_exit_depth') is not None else None,
            'wall_seconds':float(time.perf_counter()-started),'internal_wall_seconds':float(result.get('wall_seconds',0)),
            'stop_reason':str(result.get('stop_reason','unknown')),
            'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30)}
print('Loaded pinned model. No untracked smoke generations executed.')


In [ ]:
def natural_id(p):return p['problem_id']+'|natural|'+str(BASE_SEED)
def natural_run(p):
    tid=natural_id(p);old=get_rec('natural',tid)
    if old:return old
    r=invoke(encode_prompt(p),MAX_NEW,GEN_MODE,BASE_SEED)
    txt=decode_ids(r['token_ids']);v=grade_output(txt,p)
    obj={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':p['problem_id'],'split':p['split'],
         'kind':p['kind'],'problem_fingerprint':p['problem_fingerprint'],'original_output':txt,
         'original_token_ids':r.pop('token_ids'),'evaluation':v,**r}
    save_rec('natural',obj)
    return obj

def cohort_from_natural(split):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    targets=CAL_TARGET if split=='calibration' else TEST_TARGET
    bykind=defaultdict(list)
    for p in bank:
        rec=get_rec('natural',natural_id(p))
        if rec and rec['evaluation']['eligible_semantic_failure']:bykind[p['kind']].append(p['problem_id'])
    if not all(len(bykind[k])>=targets[k] for k in targets):
        print('Eligible failures:',{k:len(bykind[k]) for k in targets},'required:',targets)
        return None
    subset=[x for k in targets for x in bykind[k][:targets[k]]]
    return subset

def natural_batch(split,start,limit=8):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    if cohort_from_natural(split):print('Already enough failures; skipping further candidate generation.');return
    for p in bank[start:start+limit]:
        if cohort_from_natural(split):break
        r=natural_run(p)
        print(p['problem_id'],r['evaluation']['success'],round(r['wall_seconds'],2),'s')
    print('Progress',len(all_recs('natural')),'native generations total.')

def freeze_cohorts():
    cp=cohort_from_natural('calibration');hp=cohort_from_natural('heldout')
    assert cp is not None and hp is not None,'Not enough model-native failures yet: continue natural-generation batches.'
    assert set(cp).isdisjoint(hp)
    content={'protocol_hash':RUN_SIG,'calibration':cp,'heldout':hp,'problem_bank_sha256':BANK_HASH}
    file=ROOT/'frozen_cohorts.json'
    if file.exists():assert json.loads(file.read_text())==content,'Frozen cohort differs; refusing test-set mutation.'
    else:atomic_write(file,content)
    print('FROZEN',len(cp),'calibration and',len(hp),'heldout native failures.')
    return content

In [ ]:

print('RUN SIG:',RUN_SIG)
print('CHECKPOINT:',ROOT)
for stage in ('natural','calibration','heldout','test_primitives','staged','staged_primitives'):
    print(stage,len(all_recs(stage)))


In [ ]:

def breakdown(split):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    targets=CAL_TARGET if split=='calibration' else TEST_TARGET
    counts={k:0 for k in targets}; untried={k:0 for k in targets}
    for p in bank:
        r=get_rec('natural',natural_id(p))
        if r is None:untried[p['kind']]+=1
        elif r['evaluation']['eligible_semantic_failure']:counts[p['kind']]+=1
    return counts,{k:max(0,targets[k]-counts[k]) for k in targets},untried
for split in ('calibration','heldout'):print(split,breakdown(split))


In [ ]:
# Checkpoint-safe new candidate generation. This intentionally does not revisit 144 saved IDs.
# Alternates splits/categories and prioritizes *untried* bins with missing quotas.
MAX_NEW_IN_SESSION=120
MAX_WALL_MINUTES=65
start_clock=time.monotonic()
new=0
labels=[('calibration','arithmetic'),('heldout','arithmetic'),
        ('calibration','multistep'),('heldout','multistep'),
        ('calibration','countdown'),('heldout','countdown')]

def find_next_untried(split,kind):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    for p in bank:
        if p['kind']==kind and get_rec('natural',natural_id(p)) is None:
            return p
    return None

while new<MAX_NEW_IN_SESSION and (time.monotonic()-start_clock)<MAX_WALL_MINUTES*60:
    active=[]
    for split,kind in labels:
        counts,deficit,untried=breakdown(split)
        if deficit[kind]>0 and untried[kind]>0:
            p=find_next_untried(split,kind)
            if p is not None: active.append((split,kind,p,deficit[kind],untried[kind]))
    if not active:break
    # Fair deterministic cycling instead of sending all new generations to a single category.
    split,kind,p,_,_=active[new%len(active)]
    assert get_rec('natural',natural_id(p)) is None, 'Refusing to rerun a saved trial'
    r=natural_run(p)
    new+=1
    print(f'NEW {{new}}: {{split}}/{{kind}} {{p["problem_id"]}} '+
          f'eligible={{r["evaluation"]["eligible_semantic_failure"]}} '+
          f'stop={{r.get("stop_reason")}}',flush=True)
    if new%8==0:
        print('CHECKPOINT:',{{s:breakdown(s) for s in ('calibration','heldout')}},flush=True)

print('New generations:',new,'Total records:',len(all_recs('natural')))
for s in ('calibration','heldout'):print(s,breakdown(s))
if all(not any(breakdown(s)[1].values()) for s in ('calibration','heldout')):
    COHORTS=freeze_cohorts()
    print('COHORTS FROZEN. Calibration may begin.')
else:
    exhausted=[(s,k) for s in ('calibration','heldout') for k,x in breakdown(s)[1].items() if x>0 and breakdown(s)[2][k]==0]
    if exhausted:print('BANK EXHAUSTED for',exhausted,'; STOP: protocol revision must be separately registered.')
    else:print('Checkpoint saved; discovery not yet complete.')


In [ ]:

# Create a portable PARTIAL or COMPLETE ZIP; no trial records are changed.
import zipfile
archive=Path('/content/RepairScopeX_NB29E_Continuation_Snapshot.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in ROOT.rglob('*'):
        if f.is_file() and not f.name.endswith('.tmp'):
            z.write(f,str(f.relative_to(ROOT)))
assert zipfile.ZipFile(archive).testzip() is None
print('Snapshot:',archive,'bytes:',archive.stat().st_size)
# Optional: from google.colab import files; files.download(str(archive))
